In [13]:
!pip install -q langchain langchain-community langchain-groq langchain-text-splitters faiss-cpu sentence-transformers

In [30]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

In [31]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [32]:
!pip install -q pypdf

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('/content/Sample_RAG_Project_Document.pdf')
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = splitter.split_documents(pages)
print(f"{len(docs)} chunks created")

3 chunks created


In [33]:
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [34]:
@tool
def retrieve_documents(query: str) -> str:
    """Search the knowledge base for relevant document chunks. Use this
    whenever you need factual information to answer the user's question."""
    results = retriever.invoke(query)
    return "\n---\n".join(d.page_content for d in results)

tools = [retrieve_documents]
llm_with_tools = llm.bind_tools(tools)

In [35]:
def agentic_rag_query(user_question, max_turns=5):
    messages = [
        SystemMessage(content=(
            "You are a research assistant. Use the retrieve_documents tool "
            "whenever you need information from the knowledge base. If the "
            "first retrieval doesn't fully answer the question, rewrite the "
            "query and retrieve again before answering. If you cannot find "
            "an answer after retrying, say so clearly rather than guessing."
        )),
        HumanMessage(content=user_question)
    ]

    for turn in range(max_turns):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            if call["name"] == "retrieve_documents":
                result = retrieve_documents.invoke(call["args"])
                messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

    return "Reached max turns without a final answer."

In [38]:
answer = agentic_rag_query("what is Document Loader:")
print(answer)

Based on the information retrieved, the Document Loader is a component of the Retrieval-Augmented Generation (RAG) architecture that loads PDF, Word, or text files.


In [39]:
 def agentic_rag_with_reflection(user_question):
    draft = agentic_rag_query(user_question)

    critique_prompt = f"""Question: {user_question}
Draft answer: {draft}

Check the draft for unsupported claims. If it's solid, repeat it as-is.
If not, produce a corrected version."""

    response = llm.invoke([HumanMessage(content=critique_prompt)])
    return response.content

print(agentic_rag_with_reflection("What makes agentic RAG different from standard RAG?"))

The draft answer contains an unsupported claim. It states that the speaker was unable to find any information, but it doesn't provide any evidence or context to support this claim. 

A corrected version of the draft answer would be:

Draft answer: Unfortunately, I couldn't find any information on what makes agentic RAG different from standard RAG.

This revised version is more accurate and doesn't make an unsubstantiated claim. It simply states that the speaker couldn't find any information, which is a more neutral and honest statement.
